# OCR Corrector — Fine-tuning on Google Colab (T4)

**Model:** `google/flan-t5-base`  
**Task:** seq2seq — `correct OCR: {noisy_text}` → `{clean_text}`  
**Dataset:** 2922 pairs (2337 train / 292 val / 293 test) — real + synthetic OCR noise

> **Before starting:** `Runtime → Change runtime type → GPU (T4)`

## 1. Verify GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 2. Clone repository and install dependencies

In [ ]:
!git clone https://github.com/Cronosteak/ocr-corrector-papers.git
%cd ocr-corrector-papers

In [ ]:
!pip install -q transformers datasets pyyaml accelerate jiwer matplotlib

## 3. Upload dataset from local machine

Upload `train.json`, `val.json`, `test.json` and `synthetic_test.json` from `data/pairs/`:

In [ ]:
from google.colab import files
import os

os.makedirs("data/pairs", exist_ok=True)
uploaded = files.upload()  # select train.json, val.json, test.json, synthetic_test.json

for name, content in uploaded.items():
    with open(f"data/pairs/{name}", "wb") as f:
        f.write(content)
    print(f"Saved: data/pairs/{name}")

In [ ]:
import json
for split in ["train", "val", "test", "synthetic_test"]:
    path = f"data/pairs/{split}.json"
    if os.path.exists(path):
        d = json.load(open(path))
        print(f"{split}: {len(d)} pairs")

## 4. Train

In [ ]:
import yaml, gc, torch

with open("configs/train_config.yaml") as f:
    config = yaml.safe_load(f)

gc.collect()
torch.cuda.empty_cache()
print(f"epochs={config['num_epochs']} | batch={config['batch_size']} | grad_accum={config['gradient_accumulation_steps']} | optim={config['optim']} | fp16={config['fp16']}")

In [ ]:
!python -m src.model.train

## 5. Training curves

In [ ]:
import json
import matplotlib.pyplot as plt

history = json.load(open("models/ocr-corrector/train_history.json"))

train_steps, train_loss = [], []
eval_steps, eval_loss = [], []

for entry in history:
    if "loss" in entry and "eval_loss" not in entry:
        train_steps.append(entry["step"])
        train_loss.append(entry["loss"])
    if "eval_loss" in entry:
        eval_steps.append(entry["step"])
        eval_loss.append(entry["eval_loss"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_steps, train_loss, label="Train loss", linewidth=1.5)
ax.plot(eval_steps, eval_loss, label="Validation loss", linewidth=1.5, marker="o", markersize=4)
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Training curves — flan-t5-base OCR corrector")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("models/ocr-corrector/training_curves.png", dpi=150)
plt.show()
print("Saved: models/ocr-corrector/training_curves.png")

## 6. Evaluate — CER / WER (synthetic test set)

Evaluation on `synthetic_test.json`: pairs where the ground truth is a clean abstract sentence and the OCR input is the same sentence with simulated noise. This measures the model's actual correction ability without the semantic mismatch of real OCR pairs.

In [ ]:
!python -m src.model.evaluate --model models/ocr-corrector --data data/pairs/synthetic_test.json | tee models/ocr-corrector/eval_results.json

In [ ]:
import json
import matplotlib.pyplot as plt

results = json.load(open("models/ocr-corrector/eval_results.json"))

metrics = ["CER", "WER"]
baseline = [results["baseline_cer"], results["baseline_wer"]]
corrected = [results["cer"]["corrected"], results["wer"]["corrected"]]
improvement = [results["cer"]["relative_improvement_pct"], results["wer"]["relative_improvement_pct"]]

x = range(len(metrics))
fig, ax = plt.subplots(figsize=(7, 5))
bars1 = ax.bar([i - 0.2 for i in x], baseline, width=0.4, label="Baseline (raw OCR)", color="#e74c3c")
bars2 = ax.bar([i + 0.2 for i in x], corrected, width=0.4, label="Corrected (flan-t5)", color="#2ecc71")
ax.set_xticks(list(x))
ax.set_xticklabels(metrics, fontsize=13)
ax.set_ylabel("Error rate (lower is better)")
ax.set_title("CER / WER: Baseline vs Corrected model (synthetic test set)")
ax.legend()
ax.set_ylim(0, 1.1)
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{bar.get_height():.3f}", ha="center", fontsize=10)
for i, pct in enumerate(improvement):
    color = "#27ae60" if pct > 0 else "#c0392b"
    ax.text(i, 1.07, f"{'+' if pct > 0 else ''}{pct:.1f}%", ha="center", fontsize=9, color=color)
plt.tight_layout()
plt.savefig("models/ocr-corrector/cer_wer_comparison.png", dpi=150)
plt.show()
print("Saved: models/ocr-corrector/cer_wer_comparison.png")

## 7. CER distribution per sample

In [ ]:
import json
import matplotlib.pyplot as plt
from jiwer import cer
from src.model.predict import load_model, correct_batch

test_pairs = json.load(open("data/pairs/synthetic_test.json"))
model, tokenizer = load_model("models/ocr-corrector")

ocr_texts = [p["ocr"] for p in test_pairs]
gt_texts  = [p["ground_truth"] for p in test_pairs]
corrected_texts = correct_batch(ocr_texts, model, tokenizer, batch_size=16)

baseline_cer_per  = [cer(gt, ocr)  for ocr, gt in zip(ocr_texts, gt_texts)]
corrected_cer_per = [cer(gt, corr) for corr, gt in zip(corrected_texts, gt_texts)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].hist(baseline_cer_per,  bins=20, color="#e74c3c", edgecolor="white")
axes[0].set_title("CER distribution — Baseline (raw OCR)")
axes[0].set_xlabel("CER")
axes[0].set_ylabel("Number of samples")
axes[1].hist(corrected_cer_per, bins=20, color="#2ecc71", edgecolor="white")
axes[1].set_title("CER distribution — Corrected model")
axes[1].set_xlabel("CER")
plt.suptitle("Per-sample CER distribution (synthetic test set)", fontsize=13)
plt.tight_layout()
plt.savefig("models/ocr-corrector/cer_distribution.png", dpi=150)
plt.show()
print("Saved: models/ocr-corrector/cer_distribution.png")

## 8. Qualitative examples

In [ ]:
import json, random
from src.model.predict import load_model, correct_text

test_pairs = json.load(open("data/pairs/synthetic_test.json"))
model, tokenizer = load_model("models/ocr-corrector")

samples = random.sample(test_pairs, 5)
for i, pair in enumerate(samples, 1):
    corrected = correct_text(pair["ocr"], model=model, tokenizer=tokenizer)
    print(f"--- Example {i} ---")
    print(f"OCR:       {pair['ocr'][:120]}")
    print(f"Corrected: {corrected[:120]}")
    print(f"GT:        {pair['ground_truth'][:120]}")
    print()

## 9. Download model and figures

In [ ]:
!zip -r ocr-corrector-model.zip models/ocr-corrector/
from google.colab import files
files.download("ocr-corrector-model.zip")

# OCR Corrector — Fine-tuning en Google Colab (T4)

**Modelo:** `google/flan-t5-base`  
**Tarea:** seq2seq — `correct OCR: {texto_ocr}` → `{texto_limpio}`  
**Dataset:** 1728 pares OCR↔abstract (1382 train / 172 val / 174 test)

> **Antes de empezar:** `Entorno de ejecución → Cambiar tipo → GPU (T4)`

## 1. Verificar GPU

In [ ]:
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 2. Clonar repositorio e instalar dependencias

In [ ]:
!git clone https://github.com/Cronosteak/ocr-corrector-papers.git
%cd ocr-corrector-papers

In [ ]:
!pip install -q transformers datasets pyyaml accelerate jiwer matplotlib

## 3. Subir dataset desde local

Subir `train.json`, `val.json` y `test.json` desde `data/pairs/` de tu máquina:

In [ ]:
from google.colab import files
import os

os.makedirs('data/pairs', exist_ok=True)
uploaded = files.upload()  # seleccionar train.json, val.json, test.json

for name, content in uploaded.items():
    with open(f'data/pairs/{name}', 'wb') as f:
        f.write(content)
    print(f'Guardado: data/pairs/{name}')

In [ ]:
import json
for split in ['train', 'val', 'test']:
    d = json.load(open(f'data/pairs/{split}.json'))
    print(f'{split}: {len(d)} pares')

## 4. Entrenar

In [ ]:
import yaml, gc, torch

with open('configs/train_config.yaml') as f:
    config = yaml.safe_load(f)

gc.collect()
torch.cuda.empty_cache()
print(f"epochs={config['num_epochs']} | batch={config['batch_size']} | grad_accum={config['gradient_accumulation_steps']} | optim={config['optim']} | fp16={config['fp16']}")

In [ ]:
!python -m src.model.train

## 5. Curvas de entrenamiento

In [ ]:
import json
import matplotlib.pyplot as plt

history = json.load(open('models/ocr-corrector/train_history.json'))

train_steps, train_loss = [], []
eval_steps, eval_loss = [], []

for entry in history:
    if 'loss' in entry and 'eval_loss' not in entry:
        train_steps.append(entry['step'])
        train_loss.append(entry['loss'])
    if 'eval_loss' in entry:
        eval_steps.append(entry['step'])
        eval_loss.append(entry['eval_loss'])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_steps, train_loss, label='Train loss', linewidth=1.5)
ax.plot(eval_steps, eval_loss, label='Val loss', linewidth=1.5, marker='o', markersize=4)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Curvas de entrenamiento — flan-t5-base OCR corrector')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('models/ocr-corrector/training_curves.png', dpi=150)
plt.show()
print('Guardado: models/ocr-corrector/training_curves.png')

## 6. Evaluar (CER / WER baseline vs corregido)

In [ ]:
!python -m src.model.evaluate --model models/ocr-corrector --data data/pairs/test.json | tee models/ocr-corrector/eval_results.json

In [ ]:
import json
import matplotlib.pyplot as plt

results = json.load(open('models/ocr-corrector/eval_results.json'))

metrics = ['CER', 'WER']
baseline = [results['baseline_cer'], results['baseline_wer']]
corrected = [results['cer']['corrected'], results['wer']['corrected']]

x = range(len(metrics))
fig, ax = plt.subplots(figsize=(7, 5))
bars1 = ax.bar([i - 0.2 for i in x], baseline, width=0.4, label='Baseline (OCR)', color='#e74c3c')
bars2 = ax.bar([i + 0.2 for i in x], corrected, width=0.4, label='Corregido (flan-t5)', color='#2ecc71')
ax.set_xticks(list(x))
ax.set_xticklabels(metrics, fontsize=13)
ax.set_ylabel('Error rate (menor es mejor)')
ax.set_title('CER / WER: Baseline vs Modelo corregido')
ax.legend()
ax.set_ylim(0, 1.1)
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('models/ocr-corrector/cer_wer_comparison.png', dpi=150)
plt.show()
print('Guardado: models/ocr-corrector/cer_wer_comparison.png')

## 7. Ejemplos cualitativos

In [ ]:
import json, random
from src.model.predict import load_model, correct_text

test_pairs = json.load(open('data/pairs/test.json'))
model, tokenizer = load_model('models/ocr-corrector')

samples = random.sample(test_pairs, 5)
for i, pair in enumerate(samples, 1):
    corrected = correct_text(pair['ocr'], model=model, tokenizer=tokenizer)
    print(f'--- Ejemplo {i} ---')
    print(f"OCR:       {pair['ocr'][:120]}")
    print(f'Corregido: {corrected[:120]}')
    print(f"GT:        {pair['ground_truth'][:120]}")
    print()

## 8. Descargar modelo y gráficos

In [ ]:
!zip -r ocr-corrector-model.zip models/ocr-corrector/
from google.colab import files
files.download('ocr-corrector-model.zip')